# 11 — Simple instruments

**Theme:** instruments that report a handful of values rather than a size
distribution.

A dosimeter, a gas monitor and a photometer all produce one or a few channels
over time. They reuse the whole `Aerosol1D` machinery — activities, time
operations, statistics, plotting — and add accessors named after what they
actually measure.

Instruments that do not count particles (gas monitors, environmental loggers,
the Partector) deliberately have **no** `total_concentration`: reporting a
chlorine reading as a particle number concentration would be wrong. Use their
own accessors instead.

In [ ]:
import aerosoltools as at

## Partector — lung-deposited surface area

The Partector reports LDSA, the surface area of particles that deposits in the
lung. The TEM variant also samples particles onto grids for microscopy, and
records when it did so.

In [ ]:
partector = at.load_partector_file("../../tests/data/Sample_Partector.txt")

print("instrument:", partector.instrument)
print("columns   :", list(partector.data))
print("unit      :", partector.unit)

In [ ]:
print("LDSA :", partector.ldsa.head(3).tolist(), partector.unit)
print("flow :", partector.flow.head(3).tolist())

`tem_samples` reports the grid-sampling periods, so microscopy results can be
tied back to the time series they came from.

In [ ]:
partector.tem_samples

In [ ]:
fig, ax = partector.plot_total_conc()
ax.set_title("Partector LDSA")

## DiSCmini — number, size and LDSA

The DiSCmini reports a particle number concentration, a mean particle size and
LDSA from the same measurement. It *does* count particles, so it keeps
`total_concentration`.

In [ ]:
disc = at.load_discmini_file("../../tests/data/Sample_Discmini.txt")

print("columns:", list(disc.data))
print(f"number : {disc.total_concentration.mean():.0f} {disc.unit}")
print(f"size   : {disc.size.mean():.1f} nm")
print(f"LDSA   : {disc.ldsa.mean():.1f}")

Mean size is a single number, not a distribution — the DiSCmini cannot
distinguish a narrow distribution from a wide one with the same mean. For that
you need a sizing instrument, see [8 — PSD fitting](08-psd-fitting.ipynb).

## DustTrak — PM mass fractions

The DustTrak DRX reports the standard mass fractions simultaneously.

In [ ]:
dust = at.load_dusttrak_file("../../tests/data/Sample_DustTrak.csv")

print("instrument   :", dust.instrument)
print("serial number:", dust.serial_number)
print("columns      :", list(dust.data))
print("units        :", dust.column_units)

In [ ]:
print(f"PM1   {dust.pm1.mean():7.1f}")
print(f"PM2.5 {dust.pm2_5.mean():7.1f}")
print(f"PM4   {dust.pm4.mean():7.1f}")
print(f"PM10  {dust.pm10.mean():7.1f}")
print(f"Total {dust.total.mean():7.1f}   (all in ug/m3)")

The fractions are cumulative, so each is at least as large as the one below it.
The gaps between them describe the coarseness of the aerosol.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
for name in ["pm1", "pm2_5", "pm4", "pm10"]:
    ax.plot(dust.time, getattr(dust, name), label=name.upper().replace("_", "."))
ax.set_ylabel("Mass concentration [ug/m3]")
ax.legend()
ax.set_title("DustTrak cumulative mass fractions")

## Gas monitors

`Gas1D` covers instruments measuring one gas at a time. The reading is always
`.concentration`; which gas it is lives in `.dtype`. That split means the same
code works whatever the instrument is configured for.

A Ranger uses interchangeable heads, so one file can contain several
components — the loader returns one object per component.

In [ ]:
loaded = at.load_ranger_file("../../tests/data/Ranger_2605-1000088-A_20260612-0603.csv")
objects = loaded if isinstance(loaded, list) else [loaded]

for obj in objects:
    dtype = obj.dtype if not isinstance(obj.dtype, dict) else "PM channels"
    print(f"{type(obj).__name__:12s} {str(dtype):14s} {len(obj.data):5d} samples")

In [ ]:
chlorine = next(o for o in objects if o.dtype == "Cl₂")

print(f"gas   : {chlorine.dtype}")
print(f"unit  : {chlorine.unit}")
print(f"mean  : {chlorine.concentration.mean():.4f} {chlorine.unit}")

A Tiger VOC monitor is the same class with a different gas, so the same
accessor works unchanged.

In [ ]:
tiger = at.load_tiger_file("../../tests/data/Sample_Tiger.csv")

print(f"gas   : {tiger.dtype}")
print(f"unit  : {tiger.unit}")
print(f"mean  : {tiger.concentration.mean():.1f} {tiger.unit}")

In [ ]:
fig, ax = tiger.plot_total_conc()
ax.set_title(f"Tiger {tiger.dtype}")

## Environmental loggers

`Environmental1D` covers temperature, humidity and related channels. Only the
channels actually present in the file are available.

In [ ]:
env = at.load_fourtec_file("../../tests/data/Sample_Fourtec.xlsx")

print("columns:", list(env.data))
print(f"temperature: {env.temperature.mean():.2f} degC")
print(f"RH         : {env.rh.mean():.1f} %")

Asking for a channel the file does not contain raises a message naming what
*is* available, rather than a bare `KeyError`.

In [ ]:
try:
    env.pressure
except AttributeError as err:
    print("AttributeError:", err)

## The shared contract

Non-particle instruments raise on `total_concentration` by design.

In [ ]:
for obj, label in [(tiger, "Gas1D"), (env, "Environmental1D"),
                   (partector, "Partector")]:
    try:
        obj.total_concentration
        print(f"{label:16s} has total_concentration")
    except AttributeError:
        print(f"{label:16s} correctly has no total_concentration")

Everything else does work on them — activities, cropping, summaries, plotting —
because they are ordinary `Aerosol1D` objects underneath.

In [ ]:
tiger.mark_activities({
    "Weaving": [(str(tiger.time[100]), str(tiger.time[400]))],
})
tiger.summarize_activities()

---

**Next:** [12 — Aethalometer](12-aethalometer.ipynb).